In [ ]:
# Jupyter notebook for Session 0.
# NB! Restart kernel and clear all outputs before committing to git.

from dotenv import load_dotenv
import os
import sqlalchemy as sa
import pandas as pd
import plotly.express as px
from supabase import create_client
import kaleido # For saving Plotly figures as files.
from pathlib import Path # Filesystem tools.

In [ ]:
# Load configuration from .env file in parent directory.
# Dotenv is smart enough to search the entire directory tree for .env files.
load_dotenv(override=True)

# Connect to Supabase directly.
supabase_direct = sa.create_engine(os.getenv("SUPABASE_CONNECTION_STRING"))

# And via API, for demonstration purposes.
supabase_api = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Loading data using an SQL query over direct connection.
# This offers more flexible control over data set sizes.
def load_data_using_sql(sql):
    query = sa.text(sql)
    return pd.read_sql(query, supabase_direct)

# Loading data over the Supabase API.
# This should not be done in real-world situations when working with tables with millions of rows!
# Supabase API has a built-in limit of 1000 rows, so this method could also yield misleading results.
def load_data_using_api(table):
    response = supabase_api.table(table).select('*').execute()
    return pd.DataFrame(response.data)


In [ ]:
# Load all data from the sales table.
df = load_data_using_sql("SELECT * FROM sales")
df

In [ ]:
# Display the first 5 rows from the sales table.
df.head()

In [ ]:
# Display the number of rows x number of columns.
df.shape

In [ ]:
# Display the number of rows in the sales table.
df.shape[0]

In [ ]:
# Generate descriptive statistics for the DataFrame.
df.describe()

In [ ]:
# Display columns, data types, and count of non-NULL values.
df.info()

In [ ]:
# Define a sample dataset.
data = {
    "customer_id": [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                    1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
    "sale_date": ["2024-01-15", "2024-01-16", "2024-02-01", "2024-02-20", "2024-03-01",
                   "2024-03-05", "2024-03-15", "2024-04-10", "2024-04-12", "2024-04-20",
                   "2024-05-01", "2024-05-10", "2024-05-15", "2024-06-01", "2024-06-05",
                   "2024-06-10", "2024-06-20", "2024-07-01", "2024-07-05", "2024-07-10"],
    "total_price": [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00, 78.00, 92.00,
                     160.00, 44.00, 88.50, 230.00, 37.00, 175.00, 110.00, 65.00, 95.00, 125.00],
    "city": ["Tallinn", "Tartu", "Tallinn", "Tallinn", "Tartu", "Pärnu", "Tallinn", "Tallinn",
             "Tartu", "Pärnu", "Tartu", "Tallinn", "Tartu", "Tallinn", "Pärnu", "Pärnu",
             "Tartu", "Tallinn", "Tallinn", "Tartu"],
    "product_category": ["Dresses", "Tops", "Denim", "Accessories", "Tops", "Denim", "Tops",
                         "Dresses", "Denim", "Accessories", "Dresses", "Tops", "Denim",
                         "Dresses", "Accessories", "Denim", "Tops", "Accessories", "Dresses", "Denim"]
}

# Construct a pandas DataFrame using the given dataset.
df = pd.DataFrame(data)

In [ ]:
# Display the number of rows and columns.
df.shape

In [ ]:
# Display the first 5 rows.
df.head()

In [ ]:
# Display basic information about the DataFrame.
df.info()

In [ ]:
# Display descriptive statistics for the DataFrame.
df.describe()

In [ ]:
# Display the data types of columns.
df.dtypes

In [ ]:
# Display the number of unique customer IDs.
df["customer_id"].nunique()

In [ ]:
# Display unique city names in the DataFrame.
df["city"].unique()

In [ ]:
# Display the number of occurrences of each city in the DataFrame.
df["city"].value_counts()

In [ ]:
# Calculate the total revenue (sum of total_price for each sale).
df["total_price"].sum()

In [ ]:
# Display the number of unique product categories (similar to count(*) in PostgreSQL).
df["product_category"].nunique()

In [ ]:
# Display unique product categories (similar to SELECT DISTINCT in PostgreSQL).
df["product_category"].unique()

In [ ]:
# Display the count of values for each unique product category.
# This is similar to: SELECT product_category, count(*) AS products_count FROM products GROUP BY product_category;
df["product_category"].value_counts()

In [ ]:
# Display descriptive statistics for the 'product_category' column.
df["product_category"].describe()

In [ ]:
# Demonstrate boolean indexing.
df["total_price"] > 100

In [ ]:
# Use boolean indexing to filter the DataFrame, including only rows where 'total_price' is greater than 100.
# In SQL, this would be: SELECT * FROM sales WHERE total_price > 100;
df[df["total_price"] > 100]

In [ ]:
# Filter the DataFrame to include rows where 'city' is "Tallinn".
# SQL equivalent: WHERE city = 'Tallinn'
df[df["city"] == "Tallinn"]

In [ ]:
# Filter the pandas DataFrame by 'total_price' and 'city'.
# SQL equivalent: WHERE total_price > 100 AND city = 'Tallinn'.
df[(df["total_price"] > 100) & (df["city"] == "Tallinn")]

In [ ]:
# Summarize total revenue for each city.
# In SQL, this is achieved using GROUP BY.
grouped = df.groupby("city")["total_price"].sum()
print(type(grouped))
grouped

In [ ]:
# Use multiple aggregate functions over a single column.
grouped = df.groupby("city")["total_price"].agg(["sum", "mean", "count"])
print(type(grouped))
grouped

In [ ]:
# Note that agg(["sum"]) behaves differently from sum(). It returns a DataFrame object instead of a Series.
grouped = df.groupby("city")["total_price"].agg(["sum"])
print(type(grouped))
grouped

In [ ]:
# Summarize over multiple columns.
df.groupby(["city", "product_category"])["total_price"].sum()

In [ ]:
# Perform multiple aggregations over multiple columns.
grouped = df.groupby(["city", "product_category"])["total_price"].agg(["sum", "mean", "count", "min", "max"])
grouped.head()

In [ ]:
# Sort by total_price in descending order.
# SQL equivalent: ORDER BY total_price DESC;
df.sort_values("total_price", ascending=False)

In [ ]:
# Sort by multiple columns.
# SQL equivalent: ORDER BY city ASC, total_price DESC;
df.sort_values(["city", "total_price"], ascending=[True, False])

In [ ]:
df_sales = load_data_using_sql("SELECT * FROM sales;")
df_customers = load_data_using_sql("SELECT * FROM customers;")

df_sales.head()

In [ ]:
# Join pandas DataFrames.
# SQL equivalent:
"""
SELECT
    *
FROM sales s
LEFT JOIN customers c ON s.customer_id = s.customer_id
"""
merged = pd.merge(df_sales, df_customers, on="customer_id", how="left")
print(type(merged))
merged.head()

In [ ]:
# Check the shapes of the DataFrames.
# The number of rows in 'merged' should be the same as 'df_sales'.
# The number of columns in 'merged' should be one less than the sum of columns in 'df_sales' and 'df_customers',
# because the 'customer_id' column will not be duplicated in the merged DataFrame.
df_sales.shape, df_customers.shape, merged.shape

In [ ]:
# Add a new column to the pandas DataFrame.
df["discount"] = df["total_price"] * 0.1
df.head()

In [ ]:
# Perform conditional calculations on the 'total_price' column to add a 'segment' column.
# SQL equivalent: CASE WHEN.
df["segment"] = df["total_price"].apply(
    lambda x: "Big" if x > 100 else "Small"
)
df.head()

In [ ]:
# Convert 'sale_date' column to datetime objects.
df["sale_date"] = pd.to_datetime(df["sale_date"])
df.head()

In [ ]:
# Extract month and year from 'sale_date' and store them in new columns.
df["month"] = df["sale_date"].dt.month
df["year"] = df["sale_date"].dt.year
df.head()

In [ ]:
df_sales.head()

In [ ]:
# Add a new column to the merged DataFrame, providing business interpretation to 'total_price'.
merged["order_size"] = merged["total_price"].apply(
    lambda x: "Large (100+)" if x >= 100 else "Small (<100)"
)
merged.head()

In [ ]:
# Filter for all sales rows where the customer city is Tallinn.
tallinn = merged[merged["city"] == "Tallinn"]
tallinn.describe()

In [ ]:
# Aggregate sales by city, with cities having the largest sum appearing at the top.
city_revenue = merged.groupby("city")["total_price"].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False)
# Rename columns for better business context:
city_revenue.columns = ["total_revenue", "average_revenue", "orders"]
city_revenue.head()

In [ ]:
# Display statistics on the aggregated data.
city_revenue.describe()

In [ ]:
order_sizes_series = merged["order_size"].value_counts()
print(type(order_sizes_series))

# `reset_index()` will convert the Series to a DataFrame:
order_sizes = order_sizes_series.reset_index()
print(type(order_sizes))

order_sizes

In [ ]:
# Display top customer cities by average revenue.
city_revenue.sort_values("average_revenue", ascending=False).head(10)

In [ ]:
# Use Named Aggregation to name aggregations, an alternative to `by_customer.columns =`.
customer_summary = merged.groupby(["customer_id", "first_name", "last_name"]).agg(
    total_spending=("total_price", "sum"),
    average_order=("total_price", "mean"),
    orders=("total_price", "count")
).sort_values("total_spending", ascending=False).reset_index()

# Add a VIP status column.
customer_summary["vip_status"] = customer_summary["total_spending"].apply(
    lambda x: "YES" if x > 200 else "NO"
)

# Display the top 5 customers by total spending, including ID and name.
customer_summary.head()

In [ ]:
# Count the number of VIP customers.
customer_summary["vip_status"].value_counts().reset_index()

In [ ]:
# Create a bar chart of total revenue by city.
city_data = merged.groupby("city")["total_price"].sum().reset_index()
fig = px.bar(
    city_data,
    x="city",
    y="total_price",
    title="UrbanStyle Revenue by City",
    labels={"city": "City", "total_price": "Revenue"}
)

# Customize chart layout.
fig.update_layout(
    plot_bgcolor="white",
    font=dict(family="Calibri", size=14),
    title_font_size=18
)

# Customize axes labels.
fig.update_xaxes(title_text="City", tickangle=45)
fig.update_yaxes(title_text="Revenue (EUR)")

# Save the chart as HTML.
Path("../tmp").mkdir(exist_ok=True) # Create a directory for temporary files (should already be in .gitignore).
fig.write_html("../tmp/urbanstyle_chart.html")

# Save the chart as an image (kaleido package is used internally by Plotly for this).
fig.write_image("../tmp/urbanstyle_chart.png")

# SVG (Scalable Vector Graphics) is also supported.
# SVG images do not pixelate when zoomed in.
fig.write_image("../tmp/urbanstyle_chart.svg")

fig.show()

In [ ]:
# Create a line chart of total revenue by month.
monthly = merged.groupby(merged["sale_date"].dt.to_period("M"))["total_price"].sum().reset_index()
monthly["sale_date"] = monthly["sale_date"].astype(str)

fig = px.line(
    monthly,
    x="sale_date",
    y="total_price",
    title="UrbanStyle Monthly Revenue Trend",
    labels={"sale_date": "Month", "total_price": "Revenue (EUR)"},
    markers=True # Visualize data points as bold dots on the chart.
)
fig.show()

In [ ]:
customer_summary.head()

In [ ]:
# Create a scatter plot.
# Customer purchasing behavior: number of orders vs total spending.
fig = px.scatter(
    customer_summary,
    x="orders",
    y="total_spending",
    color="vip_status",
    title="Customers: Purchase Frequency vs Total Spending",
    labels={ "orders": "Number of Orders", "total_spending": "Total Spending (EUR)", "vip_status": "VIP Status" }
)
fig.show()

In [ ]:
df.head()

In [ ]:
# Create a pie chart.
# Show the percentage of total revenue by product category.
cat_data = df.groupby("product_category")["total_price"].sum().reset_index()
fig = px.pie(
    cat_data,
    values="total_price",
    names="product_category",
    title="Revenue Distribution by Product Category",
    labels={ "total_price": "Revenue", "product_category": "Product Category" }
)
fig.show()

In [ ]:
# Analyze revenue by product category.
cat_revenue = df.groupby("product_category")["total_price"].sum().reset_index()
cat_revenue = cat_revenue.sort_values("total_price", ascending=True)

fig = px.bar(
    cat_revenue,
    x="total_price",
    y="product_category",
    orientation="h",
    title="UrbanStyle: Revenue by Product Category",
    labels={
        "total_price": "Revenue (EUR)",
        "product_category": "Category"
    },
    text="total_price", # Display total prices on bars.
    color="product_category"
)

fig.update_layout(
    title="Revenue by Product Category",
    showlegend=False # Hide the legend.
)

fig.show()

In [ ]:
# Convert sale dates to monthly periods for aggregation.
months = merged["sale_date"].dt.to_period("M")
months

In [ ]:
# Groupby implicitly links the indexes of 'merged' and 'months', which is why the following code works.
monthly_sales = merged.groupby(months).agg(total_revenue=("total_price", "sum")).reset_index()
# Add a stringified 'month' field, as the Period type would cause an error with Plotly.
monthly_sales["month"] = monthly_sales["sale_date"].astype(str)
monthly_sales.head()

In [ ]:
fig = px.line(
    monthly_sales,
    title="Total Revenue by Month",
    x="month",
    y="total_revenue",
    labels={ "month": "Month", "total_revenue": "Total Revenue (EUR)" },
    markers=True
)

fig.update_layout(
    plot_bgcolor="white"
)

fig.show()

In [ ]:
customer_summary.head()

In [ ]:
fig = px.pie(
    customer_summary,
    title="Customer Distribution by VIP Status",
    names="vip_status",
    labels={ "vip_status": "VIP Status" }
)

fig.update_layout(
    showlegend=True
)

fig.show()

In [ ]:
# RFM analysis.
# This section performs Recency, Frequency, Monetary (RFM) analysis.

# Define the reference date for calculations.
reference_date = pd.to_datetime("2024-12-31")

# Filter all sales until the specified reference date.
df = merged[merged["sale_date"] <= reference_date]
df.info()

In [ ]:
# Recency (R): Calculate the number of days since the customer's last purchase.
recency = df.groupby("customer_id")["sale_date"].max().reset_index()
recency.columns = ["customer_id", "last_purchase"]
recency["recency_days"] = (reference_date - recency["last_purchase"]).dt.days
recency.head()

In [ ]:
# Frequency (F): Calculate the total number of purchases made by each customer.
frequency = df.groupby("customer_id").size().reset_index(name="frequency")
frequency.head()

In [ ]:
# Monetary (M): Calculate the total spending of each customer.
monetary = df.groupby("customer_id")["total_price"].sum().reset_index()
monetary.columns = ["customer_id", "monetary"]
monetary.head()

In [ ]:
# Calculate all RFM metrics at once for a more concise and simplified approach, eliminating the need for further merging.
rfm = df.groupby("customer_id").agg(
    last_purchase=("sale_date", "max"), # Recency
    number_of_purchases=("id", "size"), # Frequency - size behaves like PostgreSQL count(*)
    total_spending=("total_price", "sum") # Monetary
).reset_index()
rfm["time_since_last_purchase"] = reference_date - rfm["last_purchase"]
rfm["days_since_last_purchase"] = rfm["time_since_last_purchase"].dt.days
rfm.head()

In [ ]:
pd.qcut?

In [ ]:
# Define settings for RFM score calculations.
q = 3 # Number of quantiles.
labels = list(range(1, q + 1)) # Labels for F (Frequency) and M (Monetary).
reverse_labels = list(range(q, 0, -1)) # Labels for R (Recency) - smaller values for days_since_last_purchase are better.
labels, reverse_labels # Print out both label lists for verification.

rfm.head()

In [ ]:
# Calculate the Recency score.
rfm["R_score"] = pd.qcut(rfm["days_since_last_purchase"], q, labels=reverse_labels).astype(int)
rfm.head()

In [ ]:
# Calculate the Frequency score.
rfm["F_score"] = pd.qcut(rfm["number_of_purchases"], q, labels).astype(int)
rfm.head()

In [ ]:
# Calculate the Monetary score.
rfm["M_score"] = pd.qcut(rfm["total_spending"], q, labels).astype(int)
rfm.head()

In [ ]:
# Calculate the combined RFM score.
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]
rfm.head()

In [ ]:
# Define a function to assign customer segments based on RFM score.
def assign_segment(score):
    if score >= 8:
        return "VIP Champions"
    elif score >= 6:
        return "Loyal Customers"
    elif score >= 4:
        return "Potential Loyalists"
    else:
        return "At Risk"

rfm["segment"] = rfm["RFM_score"].apply(assign_segment)
rfm.sort_values("RFM_score", ascending=False)

In [ ]:
# Count customers and total spending by segment.
segment_counts = rfm.groupby("segment").agg(
    customers=("customer_id", "size"),
    total_spending=("total_spending", "sum")
).reset_index()
segment_counts

In [ ]:
# Visualize customer distribution by segment using a bar chart.
fig1 = px.bar(
    segment_counts,
    x="segment",
    y="customers",
    title="UrbanStyle: Customer Segment Distribution (RFM)",
    labels={"segment": "Segment", "customers": "Number of Customers"},
    color="segment"
)
fig1.show()

In [ ]:
# Visualize days since last purchase vs. total spending.
fig2 = px.scatter(
    rfm,
    x="days_since_last_purchase",
    y="total_spending",
    color="segment",
    size="number_of_purchases", # Controls the bubble size on the scatter plot.
    hover_data=["customer_id"],
    title="UrbanStyle: Recency vs Monetary (RFM)",
    labels={
        "days_since_last_purchase": "Days Since Last Purchase",
        "total_spending": "Total Spending (EUR)"
    }
)

fig2.update_layout(
    plot_bgcolor="white"
)

fig2.show()

In [ ]:
fig3 = px.pie(
    segment_counts,
    title="Customer Distribution by VIP Status",
    names="segment",
    values="total_spending",
    labels={ "segment": "Segment", "total_spending": "Total Spending" }
)

fig3.update_layout(
    showlegend=True
)

fig3.show()

In [ ]:
# Exercise 10: Calculate average purchase per city and sort by the average purchase in descending order.
sales_by_city = merged.groupby("city").agg(
    average_purchase=("total_price", "mean")
).sort_values("average_purchase", ascending=False).reset_index()
sales_by_city.head()